# McDonalds EDA
#### Autumn 2026 Obaro Erigha

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

# Load the data
facts = pd.read_csv("McD Restaurant Facts.csv", na_values=["NA", "", " "])
sales = pd.read_csv("McD Weekly Sales.csv", na_values=["NA", "", " "])

# Clean and parse
sales["wk_ending"] = pd.to_datetime(sales["wk_ending"], format="%m/%d/%y")
sales = sales.drop(columns=["urws"]) # redundant

# Merge
df = sales.merge(facts, on="REST_KEY", how="inner", validate="many_to_one")
df["revenue"] = df["totunits"] * df["wavg_price"]

print(f"Merged data shape: {df.shape}")

Merged data shape: (302400, 29)


#### Q1: Does every restaurant in the sales dataset have information in the restaurant facts dataset and vice versa? If not, describe the mismatch

In [3]:
sales_keys = set(sales["REST_KEY"])
facts_keys = set(facts["REST_KEY"])

missing_sales = facts_keys - sales_keys
missing_facts = sales_keys - facts_keys

print(f"Keys in Facts but not in Sales: {missing_sales}")
print(f"Keys in Sales but not in Facts: {missing_facts}")

if len(missing_sales) == 0 and len(missing_facts) == 0:
    print("1-to-1 match. No missing restaurants.")

Keys in Facts but not in Sales: set()
Keys in Sales but not in Facts: set()
1-to-1 match. No missing restaurants.


#### Q2: Are the menu item numbers in 1-to-1 correspondence with the item descriptions? If not, describe the many-to-one or one-to-many matches.

In [4]:
item_check = sales.groupby("ItemN")["itemdesc"].nunique()
print(item_check.value_counts())
print("\nAny ItemN with more than 1 itemdesc?")
print(item_check[item_check > 1])

itemdesc
1    25
Name: count, dtype: int64

Any ItemN with more than 1 itemdesc?
Series([], Name: itemdesc, dtype: int64)


#### Q3: What is the hierarchy for the menu items? Describe which categories flow into other categories, and how you learned this information.

In [5]:
# Show the hierarchy
print(sales[["ItemN", "itemdesc"]].drop_duplicates().sort_values("ItemN").head(30).to_string(index=False))

 ItemN                      itemdesc
     1 Total All Day Breakfast Items
     2      Total Breakfast Sandwich
     3               Total McMuffins
     4                  Egg McMuffin
     5              Sausage McMuffin
     6          Sausage Egg McMuffin
     7                Total Biscuits
     8            Bac Egg Ch Biscuit
     9               Sausage Biscuit
    10           Sausage Egg Biscuit
    11                     Hashbrown
    12              Total McGriddles
    13          Bac Egg Ch McGriddle
    14          Sau Egg Ch McGriddle
    15                  Total Bagels
    16                 Beef Category
    17                 Large Burgers
    18                       Big Mac
    19               Core QP and QPC
    20                     Hamburger
    21                  Cheeseburger
    22              Chicken Category
    23                   Total Fries
    24                     McChicken
    25                      McDouble


#### Q4: What column or columns should we use to measure weekly store revenues?


In [6]:
# Already created in setup: df["revenue"] = df["totunits"] * df["wavg_price"]
df[["totunits", "wavg_price", "revenue"]].describe().T

,count,mean,std,min,25%,50%,75%,max
totunits,297638.0,1385.219404,1687.315560,3.00,265.00,646.00,1789.00,13217.00
wavg_price,297638.0,2.253532,0.989593,0.51,1.25,2.23,2.99,5.53
revenue,297638.0,2602.835615,2945.315214,10.17,576.18,1299.00,3653.04,21808.05


#### Q5: Do we have positive total sales for all stores in all weeks?

In [ ]:
store_week_sales = df.groupby(["REST_KEY", "wk_ending"])["revenue"].sum().reset_index()
print(f"Minimum weekly revenue: ${store_week_sales['revenue'].min():.2f}")
print(f"Any zero or negative sales? {(store_week_sales['revenue'] <= 0).any()}")

Minimum weekly revenue: $0.00
Any zero or negative sales? True


: 

#### Q6: Does each restaurant-week pair found in the data have positive sales of all menu items, or are some items only found in certain weeks or certain stores?

In [ ]:
# Count unique items per store-week
items_per_store_week = df.groupby(["REST_KEY", "wk_ending"])["ItemN"].nunique()

print(f"Total unique items in dataset: {df['ItemN'].nunique()}")
print(f"Average items sold per store-week: {items_per_store_week.mean():.1f}")
print(f"Max items sold in any store-week: {items_per_store_week.max()}")
print(f"Min items sold in any store-week: {items_per_store_week.min()}")

# Visualize sparsity
items_per_store_week.hist(bins=30)
plt.title("Number of Unique Items Sold per Store-Week")
plt.xlabel("Number of Items")
plt.ylabel("Frequency")
plt.show()
